In [1]:
import os,pickle,warnings,numpy as np,pandas as pd,matplotlib.pyplot as plt,networkx as nx
from pathlib import Path
from scipy.stats import rankdata
from sklearn.covariance import LedoitWolf
from sklearn.metrics import adjusted_mutual_info_score
from networkx.algorithms.community import louvain_communities,modularity
warnings.filterwarnings("ignore")

OUT=Path("match_SC_mask");OUT.mkdir(exist_ok=True)
ST=["Stage I","Stage II","Stage III"];ORDER=["I-II","I-III","II-III"]
PAIRS=[("I-II",ST[0],ST[1]),("I-III",ST[0],ST[2]),("II-III",ST[1],ST[2])]
BASE=["Pearson","Spearman","Partial-LW","MutualInfo"];METHODS=[m+"-SC" for m in BASE]+["NEXUS"]
DENS=[5,7.5,10,12.5,15];NSEED=50;MI_MAX_T=100000;MI_Q=5;SEED=42;MAX_T=None

PKL={ST[0]:"./result/wo_layer/stage1/binned_traces_stage_1.pkl",
     ST[1]:"./result/wo_layer/stage2/binned_traces_stage_2.pkl",
     ST[2]:"./result/wo_layer/stage3/binned_traces_stage_3.pkl"}

NP={ST[0]:("./result/wo_layer/stage1/tuning/bs32_dp0.1_sw1.0/dynamic_full_graphs.npy","./result/wo_layer/stage1/df_scaled.csv"),
    ST[1]:("./result/wo_layer/stage2/tuning/bs32_dp0.1_sw1.0/dynamic_full_graphs.npy","./result/wo_layer/stage1/df_scaled.csv"),
    ST[2]:("./result/wo_layer/stage3/tuning/bs32_dp0.1_sw0.5/dynamic_full_graphs.npy","./result/wo_layer/stage1/df_scaled.csv")}

def loadN(g,l):
    A=np.load(g).mean(0);L=pd.read_csv(l).iloc[:,0].astype(str).str.strip().values;np.fill_diagonal(A,0);return A,L
raw={s:loadN(*NP[s]) for s in ST}
common=[r for r in raw[ST[0]][1] if all(r in set(raw[s][1]) for s in ST)]

def reorder(A,L):
    d={r:i for i,r in enumerate(L)};i=[d[r] for r in common];return A[np.ix_(i,i)]
NEX={s:reorder(*raw[s]) for s in ST}

def getX(p):
    with open(p,"rb") as f:b=pickle.load(f)["binned_traces"]
    miss=[r for r in common if r not in b]
    if miss:raise ValueError(f"Missing regions: {miss}")
    X=np.nan_to_num(np.array([np.asarray(b[r],float) for r in common]))
    if MAX_T and X.shape[1]>MAX_T:X=X[:,np.linspace(0,X.shape[1]-1,MAX_T).astype(int)]
    if np.any(np.isclose(X.std(1),0)):raise ValueError(f"Zero-variance region: {p}")
    return X

XR={s:getX(PKL[s]) for s in ST}
for s in ST:print(s,XR[s].shape)

def prec2pc(Q):
    d=np.sqrt(np.clip(np.diag(Q),1e-15,None));A=-Q/np.outer(d,d);np.fill_diagonal(A,0);return np.clip(A,-1,1)

RAW={m:{} for m in BASE}
for s in ST:
    X=XR[s];Z=((X-X.mean(1,keepdims=True))/(X.std(1,keepdims=True)+1e-12)).T
    A=np.corrcoef(X);np.fill_diagonal(A,0);RAW["Pearson"][s]=A
    A=np.corrcoef(rankdata(X,axis=1));np.fill_diagonal(A,0);RAW["Spearman"][s]=A
    RAW["Partial-LW"][s]=prec2pc(LedoitWolf().fit(Z).precision_)

rng=np.random.default_rng(SEED);MI_N=min(MI_MAX_T,min(XR[s].shape[1] for s in ST))
XM={s:XR[s][:,np.sort(rng.choice(XR[s].shape[1],MI_N,replace=False))] for s in ST}
EDGES=[]
for r in range(len(common)):
    z=np.concatenate([XM[s][r][XM[s][r]!=0] for s in ST])
    EDGES.append(np.unique(np.quantile(z,np.arange(1,MI_Q)/MI_Q)) if len(z)>1 else np.array([]))

def discretize(X):
    B=np.zeros(X.shape,dtype=np.int16)
    for r,e in enumerate(EDGES):
        m=X[r]!=0;B[r,m]=np.digitize(X[r,m],e)+1
    return B

def nmi(x,y):
    k=max(int(x.max()),int(y.max()))+1;c=np.bincount(x*k+y,minlength=k*k).reshape(k,k).astype(float)
    p=c/c.sum();px=p.sum(1);py=p.sum(0);d=px[:,None]*py[None,:];m=p>0
    mi=np.sum(p[m]*np.log(p[m]/d[m]));hx=-np.sum(px[px>0]*np.log(px[px>0]));hy=-np.sum(py[py>0]*np.log(py[py>0]))
    return 2*mi/(hx+hy) if hx+hy else 0

def mi_matrix(B):
    n=len(B);A=np.zeros((n,n))
    for i in range(n):
        for j in range(i+1,n):A[i,j]=A[j,i]=nmi(B[i],B[j])
    return A

for s in ST:RAW["MutualInfo"][s]=mi_matrix(discretize(XM[s]))

sc=pd.read_excel("../SEU_net_wo_corticalLayers.xlsx",header=1,index_col=1).iloc[:,1:]
sc.index=sc.index.astype(str).str.strip();sc.columns=sc.columns.astype(str).str.strip()
SC=sc.loc[common,common].astype(float).values;np.fill_diagonal(SC,0)
DM=SC>0;SUP=DM|DM.T;np.fill_diagonal(SUP,False);OFF=~np.eye(len(common),dtype=bool)

FC={"NEXUS":NEX}
for m in BASE:FC[m+"-SC"]={s:RAW[m][s]*DM for s in ST}

ret=[]
for m in BASE:
    for s in ST:
        A=RAW[m][s];ret.append([m+"-SC",s,100*np.abs(A[DM]).sum()/np.abs(A[OFF]).sum()])
for s in ST:ret.append(["NEXUS",s,100.])
ret=pd.DataFrame(ret,columns=["method","stage","magnitude_on_SC"])

def sim(A,B):return np.corrcoef(A[DM],B[DM])[0,1]
def change(A,B,p=5):
    x=np.abs(A[DM]-B[DM]);k=max(1,int(np.ceil(len(x)*p/100)));return np.sort(x)[-k:].sum()/x.sum()

def UW(A):
    W=(np.abs(A)+np.abs(A.T))/2;np.fill_diagonal(W,0);return W

def sv(W):
    i=np.triu_indices_from(W,1);m=SUP[i];return i,m,W[i][m]

def topmask(W,p):
    _,_,x=sv(W);k=max(1,int(np.ceil(len(x)*p/100)));q=np.argpartition(x,-k)[-k:]
    z=np.zeros(len(x),bool);z[q]=1;return z

def preserve(A,B,p):
    a,b=topmask(UW(A),p),topmask(UW(B),p);obs=(a&b).sum()/a.sum();exp=a.mean()
    return 100*(obs-exp)/(1-exp) if exp<1 else 100

def prop(W,p):
    i,m,x=sv(W);ii,jj=i[0][m],i[1][m];k=max(1,int(np.ceil(len(x)*p/100)));q=np.argpartition(x,-k)[-k:]
    A=np.zeros_like(W);A[ii[q],jj[q]]=x[q];return A+A.T

def louv(W):
    G=nx.from_numpy_array(W);best=None;qb=-np.inf
    for seed in range(NSEED):
        c=louvain_communities(G,weight="weight",seed=seed);q=modularity(G,c,weight="weight")
        if q>qb:best,qb=c,q
    z=np.empty(len(W),int)
    for k,c in enumerate(best):z[list(c)]=k
    return z

res=pd.DataFrame([[m,c,sim(FC[m][a],FC[m][b]),100*change(FC[m][a],FC[m][b])]
                  for m in METHODS for c,a,b in PAIRS],
                 columns=["method","comparison","similarity","top5_change"])

edge=pd.DataFrame([[m,c,p,preserve(FC[m][a],FC[m][b],p)]
                   for m in METHODS for c,a,b in PAIRS for p in DENS],
                  columns=["method","comparison","density","preservation"])

cc=[]
for p in DENS:
    for m in METHODS:
        lab={s:louv(prop(UW(FC[m][s]),p)) for s in ST}
        cc += [[m,c,p,adjusted_mutual_info_score(lab[a],lab[b])] for c,a,b in PAIRS]
comm=pd.DataFrame(cc,columns=["method","comparison","density","AMI"])

for d,n in [(res,"benchmark_summary"),(edge,"benchmark_edge_preservation"),
            (comm,"benchmark_community"),(ret,"benchmark_mask_retention")]:
    d.to_csv(OUT/f"{n}.csv",index=False)

print("\nSimilarity\n",res.pivot(index="method",columns="comparison",values="similarity").loc[METHODS,ORDER].round(3))
print("\nTop5 change (%)\n",res.pivot(index="method",columns="comparison",values="top5_change").loc[METHODS,ORDER].round(1))
print("\nChance-corrected edge preservation (%)\n",edge.groupby(["method","density"]).preservation.mean().unstack().loc[METHODS].round(1))
print("\nMean AMI\n",comm.groupby(["method","density"]).AMI.mean().unstack().loc[METHODS].round(3))
print("\nOriginal FC magnitude on SC (%)\n",ret.pivot(index="method",columns="stage",values="magnitude_on_SC").loc[METHODS,ST].round(1))

plt.rcParams.update({"font.size":8,"axes.labelsize":9,"xtick.labelsize":8,"ytick.labelsize":8,
                     "axes.linewidth":.8,"pdf.fonttype":42,"ps.fonttype":42})
COL={"Pearson-SC":"#64748B","Spearman-SC":"#819CB4","Partial-LW-SC":"#568E7C",
     "MutualInfo-SC":"#88738F","NEXUS":"#C25F3D"}

fig=plt.figure(figsize=(12.6,7))
gs=fig.add_gridspec(2,3,left=.06,right=.985,top=.96,bottom=.10,wspace=.28,hspace=.34)
A,B,C,D,E,F=[fig.add_subplot(gs[i,j]) for i,j in [(0,0),(0,1),(0,2),(1,0),(1,1),(1,2)]]

def panel(ax,s):ax.text(-.10,1.04,s,transform=ax.transAxes,fontsize=13,fontweight="bold")
def clean(ax):ax.spines[["top","right"]].set_visible(False);ax.tick_params(length=3)
def heat(ax,df,vmin,vmax,cmap,fmt,suf=""):
    im=ax.imshow(df.values,aspect="auto",vmin=vmin,vmax=vmax,cmap=cmap)
    ax.set_yticks(range(len(df)),df.index);ax.set_xticks(range(df.shape[1]),df.columns)
    for i in range(len(df)):
        for j in range(df.shape[1]):
            v=df.iloc[i,j];ax.text(j,i,f"{v:{fmt}}{suf}",ha="center",va="center",fontsize=7,
                                   color="white" if (v-vmin)/(vmax-vmin)>.62 else "#222")
    for x in ax.spines.values():x.set_visible(False)
    return im

panel(A,"A");A.axis("off");A.set(xlim=(0,10),ylim=(0,10))
desc={"Pearson-SC":"Pearson + SC support","Spearman-SC":"Spearman + SC support",
      "Partial-LW-SC":"Partial-LW + SC support","MutualInfo-SC":"MutualInfo + SC support",
      "NEXUS":"Anatomy-constrained dynamic graph"}
for i,m in enumerate(METHODS):
    y=8.7-i*1.65;A.scatter(.8,y,s=75,color=COL[m]);A.text(1.35,y+.18,m,fontsize=9,fontweight="bold",va="center")
    A.text(1.35,y-.35,desc[m],fontsize=7.5,color="#666",va="center")
A.text(.8,.35,"Same signals and same SC-supported candidate edges.",fontsize=7.3,color="#555")

panel(B,"B")
x=res.pivot(index="method",columns="comparison",values="similarity").loc[METHODS,ORDER];x.columns=["I–II","I–III","II–III"]
im=heat(B,x,-1,1,"RdBu_r",".2f");B.set_title("Cross-stage FC similarity",fontsize=9,pad=8)
fig.colorbar(im,ax=B,fraction=.035,pad=.025,label="Correlation within SC support")

panel(C,"C")
for m in METHODS:
    z=edge[edge.method==m].groupby("density").preservation.agg(["mean","min","max"]).reindex(DENS)
    C.fill_between(DENS,z["min"],z["max"],color=COL[m],alpha=.07)
    C.plot(DENS,z["mean"],"-o",color=COL[m],lw=2.3 if m=="NEXUS" else 1.35,ms=3.5,label=m)
C.axhline(0,color="#999",lw=.8,ls="--")
C.set(xlabel="Retained strongest SC-supported edges (%)",ylabel="Chance-corrected preservation (%)")
C.set_xticks(DENS);clean(C);C.legend(frameon=False,fontsize=6.6,ncol=2)

panel(D,"D")
for m in METHODS:
    z=comm[comm.method==m].groupby("density").AMI.agg(["mean","min","max"]).reindex(DENS)
    D.fill_between(DENS,z["min"],z["max"],color=COL[m],alpha=.07)
    D.plot(DENS,z["mean"],"-o",color=COL[m],lw=2.3 if m=="NEXUS" else 1.35,ms=3.5,label=m)
D.set(xlabel="Retained strongest SC-supported edges (%)",ylabel="Community preservation (AMI)",ylim=(0,1.02))
D.set_xticks(DENS);clean(D);D.legend(frameon=False,fontsize=6.6,ncol=2)

panel(E,"E")
x=res.pivot(index="method",columns="comparison",values="top5_change").loc[METHODS,ORDER];x.columns=["I–II","I–III","II–III"]
im=heat(E,x,0,100,"magma",".1f","%");E.set_title("Change captured by top 5% SC edges",fontsize=9,pad=8)
fig.colorbar(im,ax=E,fraction=.035,pad=.025,label="% total change within SC support")

panel(F,"F")
x=ret.pivot(index="method",columns="stage",values="magnitude_on_SC").loc[METHODS,ST]
im=heat(F,x,0,100,"cividis",".1f","%");F.set_title("Original FC magnitude retained by SC mask",fontsize=9,pad=8)
fig.colorbar(im,ax=F,fraction=.035,pad=.025,label="Magnitude on directed SC support (%)")
F.text(.99,-.15,"NEXUS = 100% by construction",transform=F.transAxes,ha="right",fontsize=6.8,color="#555")

plt.savefig(OUT/"benchmark_main.png",dpi=600,bbox_inches="tight")
plt.savefig(OUT/"benchmark_main.pdf",bbox_inches="tight")
plt.close()

Stage I (89, 1000)
Stage II (89, 1000)
Stage III (89, 1000)

Similarity
 comparison      I-II  I-III  II-III
method                             
Pearson-SC     0.690  0.608   0.483
Spearman-SC    0.364  0.348   0.276
Partial-LW-SC  0.323  0.189   0.171
MutualInfo-SC  0.563  0.651   0.557
NEXUS          0.601  0.579   0.579

Top5 change (%)
 comparison     I-II  I-III  II-III
method                            
Pearson-SC     16.9   14.3    12.1
Spearman-SC    17.9   10.4    10.1
Partial-LW-SC  16.1   16.2    16.2
MutualInfo-SC  16.2   12.5    11.7
NEXUS          39.0   33.8    35.9

Chance-corrected edge preservation (%)
 density        5.0   7.5   10.0  12.5  15.0
method                                     
Pearson-SC     47.3  54.0  54.1  53.6  62.9
Spearman-SC    25.3  37.1  40.0  49.3  52.0
Partial-LW-SC  15.5  22.5  21.7  20.3  22.8
MutualInfo-SC  42.9  46.3  51.8  51.2  54.1
NEXUS          61.6  58.6  63.5  66.7  68.3

Mean AMI
 density         5.0    7.5    10.0   12.5   15.0
met

In [2]:
# ================= FIGURE =================
from matplotlib.patches import FancyBboxPatch,FancyArrowPatch
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

plt.rcParams.update({"font.size":8,"axes.labelsize":9,"xtick.labelsize":8,"ytick.labelsize":8,
                     "axes.linewidth":.8,"pdf.fonttype":42,"ps.fonttype":42})
COL={"Pearson-SC":"#64748B","Spearman-SC":"#819CB4","Partial-LW-SC":"#568E7C",
     "MutualInfo-SC":"#88738F","NEXUS":"#C25F3D"}
CB,CN="#65758B","#C25F3D";PLOT_METHODS=METHODS

fig=plt.figure(figsize=(12.6,7))
gs=fig.add_gridspec(2,3,left=.06,right=.985,top=.96,bottom=.10,wspace=.35,hspace=.30)
A,B,C,D,E,F=[fig.add_subplot(gs[i,j]) for i,j in [(0,0),(0,1),(0,2),(1,0),(1,1),(1,2)]]

def panel(ax,s):ax.text(-.09,1.035,s,transform=ax.transAxes,fontsize=13,fontweight="bold")
def clean(ax):ax.spines[["top","right"]].set_visible(False);ax.tick_params(length=3)
def box(ax,x,y,w,h,t,ec="#777",fc="#FAFAFA",fs=7.6,weight="normal"):
    ax.add_patch(FancyBboxPatch((x,y),w,h,boxstyle="round,pad=.035,rounding_size=.08",fc=fc,ec=ec,lw=.85))
    ax.text(x+w/2,y+h/2,t,ha="center",va="center",fontsize=fs,fontweight=weight,linespacing=1.18)
def arr(ax,x1,y1,x2,y2):
    ax.add_patch(FancyArrowPatch((x1,y1),(x2,y2),arrowstyle="-|>",mutation_scale=8,lw=.8,color="#777"))
def heat(ax,df,vmin,vmax,cmap,fmt,suf="",fs=7.8):
    cm=plt.get_cmap(cmap);norm=Normalize(vmin=vmin,vmax=vmax);nr,nc=df.shape
    for i in range(nr):
        for j in range(nc):
            v=df.iloc[i,j];c=cm(norm(v));r,g,b,_=c;lum=.2126*r+.7152*g+.0722*b
            ax.add_patch(plt.Rectangle((j-.5,i-.5),1,1,facecolor=c,edgecolor="none"))
            ax.text(j,i,f"{v:{fmt}}{suf}",ha="center",va="center",fontsize=fs,fontweight="bold",
                    color="black" if lum>.55 else "white")
    ax.set(xlim=(-.5,nc-.5),ylim=(nr-.5,-.5))
    ax.set_yticks(range(nr),df.index);ax.set_xticks(range(nc),df.columns)
    for q in ax.spines.values():q.set_visible(False)
    return ScalarMappable(norm=norm,cmap=cm)

# ===== C data =====
EDGE_DENS=[2.5,5,7.5,10]
edge_cc=pd.DataFrame([[m,c,p,preserve(FC[m][a],FC[m][b],p)]
                      for m in METHODS for c,a,b in PAIRS for p in EDGE_DENS],
                     columns=["method","comparison","density","preservation"])
print("\nChance-corrected edge preservation (%)")
print(edge_cc.groupby(["method","density"]).preservation.mean().unstack().loc[METHODS].round(1))

# ================= A =================
panel(A,"A");A.axis("off");A.set(xlim=(0,10),ylim=(0,10))

A.text(.2,9.25,"SC-matched conventional FC",fontsize=9,fontweight="bold",color=CB)
box(A,.25,6.70,1.85,1.05,"Regional\npopulation signals");arr(A,2.18,7.22,2.65,7.22)
A.add_patch(FancyBboxPatch((2.75,5.75),3.55,2.85,boxstyle="round,pad=.04,rounding_size=.08",
                           fc="#F7F8FA",ec=CB,lw=.9))

for y,m,d,k in [(8,"Pearson","linear dependence","Pearson-SC"),
                (7.38,"Spearman","rank dependence","Spearman-SC"),
                (6.76,"Partial-LW","conditional dependence","Partial-LW-SC"),
                (6.14,"Mutual information","nonlinear dependence","MutualInfo-SC")]:
    A.scatter(3.05,y,s=23,color=COL[k],zorder=3)
    A.text(3.27,y,m,fontsize=7.1,fontweight="bold",va="center")
    A.text(4.75,y,d,fontsize=6.5,color="#666",va="center")

arr(A,6.40,7.22,6.78,7.22)
box(A,6.88,6.65,1.30,1.15,"Binary SC\nsupport mask",ec=CB,fc="#F3F6F9",fs=7)
box(A,6.88,8.15,1.30,.75,"Structural\nconnectome",ec=CB,fc="#F3F6F9",fs=6.8)
arr(A,7.53,8.10,7.53,7.84);arr(A,8.28,7.22,8.60,7.22)
box(A,8.70,6.65,1.10,1.15,"SC-matched\nFC",ec=CB,fc="#F3F6F9",fs=7)

A.text(.2,4.45,"NEXUS",fontsize=9,fontweight="bold",color=CN)
box(A,.35,2.65,2.05,1.10,"Regional\npopulation signals")
box(A,.35,.95,2.05,1.10,"Structural\nconnectome")
arr(A,2.5,3.20,3.10,2.80);arr(A,2.5,1.50,3.10,2.25)
box(A,3.20,1.55,4.15,2.05,"NEXUS\nSC-constrained · dynamic · directed",
    ec=CN,fc="#FFF7F3",fs=7.5,weight="bold")
arr(A,7.47,2.58,8.12,2.58)
box(A,8.22,1.95,1.45,1.25,"FC\ndirected",ec=CN,fc="#FFF7F3",fs=7.5)
A.text(.35,.25,"Same signals and same anatomically admissible edge space across methods.",fontsize=6.8,color="#666")

# ================= B =================
panel(B,"B")
x=res.pivot(index="method",columns="comparison",values="similarity").loc[PLOT_METHODS,ORDER]
x.columns=["I–II","I–III","II–III"]
im=heat(B,x,0,1,"YlGnBu",".2f")
B.set_title("Cross-stage FC similarity",fontsize=9,pad=8)
fig.colorbar(im,ax=B,fraction=.035,pad=.025,label="Correlation within SC support")

# ================= C =================
panel(C,"C")
for m in METHODS:
    z=edge_cc[edge_cc.method==m].groupby("density").preservation.agg(["mean","min","max"]).reindex(EDGE_DENS)
    C.fill_between(EDGE_DENS,z["min"],z["max"],color=COL[m],alpha=.07,lw=0)
    C.plot(EDGE_DENS,z["mean"],"-o",color=COL[m],lw=2.3 if m=="NEXUS" else 1.35,ms=3.5,label=m)
C.axhline(0,color="#AAA",ls="--",lw=.7)
C.set(xlabel="Retained strongest SC-supported edges (%)",
      ylabel="Chance-corrected\nedge preservation (%)",
      xticks=EDGE_DENS,ylim=(min(-5,edge_cc.preservation.min()-5),100))
clean(C);C.legend(frameon=False,fontsize=6.6,ncol=2,loc="best")

# ================= D =================
panel(D,"D")
for m in METHODS:
    z=comm[comm.method==m].groupby("density").AMI.agg(["mean","min","max"]).reindex(DENS)
    D.fill_between(DENS,z["min"],z["max"],color=COL[m],alpha=.07,lw=0)
    D.plot(DENS,z["mean"],"-o",color=COL[m],lw=2.3 if m=="NEXUS" else 1.35,ms=3.5,label=m)
D.set(xlabel="Retained strongest SC-supported edges (%)",ylabel="Community preservation (AMI)",ylim=(0,1.02))
D.set_xticks(DENS);clean(D);D.legend(frameon=False,fontsize=6.6,ncol=2,loc="best")

# ================= E =================
panel(E,"E")
x=res.pivot(index="method",columns="comparison",values="top5_change").loc[PLOT_METHODS,ORDER]
x.columns=["I–II","I–III","II–III"]
im=heat(E,x,0,100,"magma",".1f","%")
E.set_title("Change captured by top 5% SC edges",fontsize=12,pad=8)
fig.colorbar(im,ax=E,fraction=.035,pad=.025,label="% total change within SC support")

# ================= F =================
panel(F,"F")
x=ret.pivot(index="method",columns="stage",values="magnitude_on_SC").loc[PLOT_METHODS,ST]
x.columns=["Stage I","Stage II","Stage III"]
im=heat(F,x,0,100,"cividis",".1f","%")
F.set_title("FC magnitude within SC support",fontsize=12,pad=8)
fig.colorbar(im,ax=F,fraction=.035,pad=.025,label="% absolute FC magnitude")
F.text(.99,-.14,"Conventional FC measured before masking; NEXUS = 100% by construction",
       transform=F.transAxes,ha="right",fontsize=6.8,color="#555")

plt.savefig(OUT/"benchmark_main.png",dpi=600,bbox_inches="tight",facecolor="white")
plt.savefig(OUT/"benchmark_main.pdf",bbox_inches="tight",facecolor="white")
plt.close()


Chance-corrected edge preservation (%)
density        2.5   5.0   7.5   10.0
method                               
Pearson-SC     35.9  47.3  54.0  54.1
Spearman-SC    18.8  25.3  37.1  40.0
Partial-LW-SC  16.6  15.5  22.5  21.7
MutualInfo-SC  25.2  42.9  46.3  51.8
NEXUS          63.7  61.6  58.6  63.5


In [4]:
TRAD = ["Pearson-SC", "Spearman-SC", "Partial-LW-SC", "MutualInfo-SC"]

# ===== C: chance-corrected edge preservation =====
c = (
    edge_cc
    .groupby(["method", "density"])["preservation"]
    .mean()
    .unstack(0)
)

fold_C = [
    c.loc[den, "NEXUS"] / c.loc[den, m]
    for den in c.index
    for m in TRAD
    if m in c.columns
    and np.isfinite(c.loc[den, m])
    and c.loc[den, m] > 0
]

print(
    f"C: NEXUS = {min(fold_C):.2f}–{max(fold_C):.2f}× SC-matched conventional methods"
)

# ===== D: community preservation (AMI) =====
d = (
    comm
    .groupby(["method", "density"])["AMI"]
    .mean()
    .unstack(0)
)

fold_D = [
    d.loc[den, "NEXUS"] / d.loc[den, m]
    for den in d.index
    for m in TRAD
    if m in d.columns
    and np.isfinite(d.loc[den, m])
    and d.loc[den, m] > 0
]

print(
    f"D: NEXUS = {min(fold_D):.2f}–{max(fold_D):.2f}× SC-matched conventional methods"
)

C: NEXUS = 1.09–3.98× SC-matched conventional methods
D: NEXUS = 1.16–2.26× SC-matched conventional methods


In [5]:
for name,x in [("C",c),("D",d)]:
    print(f"\n{name}")
    for m in TRAD:
        f=(x["NEXUS"]/x[m]).replace([np.inf,-np.inf],np.nan).dropna()
        f=f[f>0]
        print(f"{m:10s}: {f.min():.2f}–{f.max():.2f}×")


C
Pearson-SC: 1.09–1.78×
Spearman-SC: 1.58–3.39×
Partial-LW-SC: 2.60–3.98×
MutualInfo-SC: 1.23–2.53×

D
Pearson-SC: 1.16–1.75×
Spearman-SC: 1.24–2.24×
Partial-LW-SC: 1.60–2.26×
MutualInfo-SC: 1.19–2.02×


In [6]:
n = len(common)

n_all = n * (n - 1) // 2
n_sc = np.triu(SUP, 1).sum()

print(f"All possible undirected pairs: {n_all}")
print(f"SC-supported undirected pairs: {n_sc}")
print(f"SC support density: {100*n_sc/n_all:.2f}%")

for p in DENS:
    k_old = int(np.ceil(n_sc * p / 100))
    k_new = int(np.ceil(n_all * p / 100))

    print(
        f"Top {p:4.1f}% | "
        f"Old: {k_old:4d} edges | "
        f"New: {k_new:4d} edges | "
        f"New SC coverage: {100*k_new/n_sc:.1f}%"
    )

All possible undirected pairs: 3916
SC-supported undirected pairs: 625
SC support density: 15.96%
Top  5.0% | Old:   32 edges | New:  196 edges | New SC coverage: 31.4%
Top  7.5% | Old:   47 edges | New:  294 edges | New SC coverage: 47.0%
Top 10.0% | Old:   63 edges | New:  392 edges | New SC coverage: 62.7%
Top 12.5% | Old:   79 edges | New:  490 edges | New SC coverage: 78.4%
Top 15.0% | Old:   94 edges | New:  588 edges | New SC coverage: 94.1%
